# Feature Extraction, Engineering and Modelling

In [1]:
# Setup
from datasets import load_dataset
import numpy as np
import pandas as pd
import re
import os
import matplotlib.pyplot as plt
import seaborn as sns

/Users/mattie/Desktop/Masters/Data Science/DataScience2026/env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load token
with open("../token.txt", "r") as f:
    token = f.read().strip()
    
# Load data from hf
ds = load_dataset("MattiWhen/filtered_openfood", token = token)

df = ds["train"].to_pandas()
df.head()

,product_name,brands,nova_group,nutriscore_score,nutriscore_grade,lang,ingredients_original_tags,code,product_name_clean,product_ID
0,"[{'lang': 'main', 'text': 'Peach Sunrise'}, {'...",Once Upon a Farm,1.0,7.0,d,en,"[en:peach-puree, en:coconut-milk, en:apple, en...",0810003517586,Peach Sunrise,Once Upon a Farm_Peach Sunrise
1,"[{'lang': 'main', 'text': 'Ayvar - Roasted Red...",Granny's Secret,3.0,4.0,c,en,"[en:red-bell-pepper, en:sunflower-oil, en:salt...",8606102785030,Ayvar - Roasted Red Pepper Spread,Granny's Secret_Ayvar - Roasted Red Pepper Spread
2,"[{'lang': 'main', 'text': 'Štapići punjeni kik...","Marbo, Pardon, Pepsico",4.0,18.0,d,en,"[en:wheat-flour-min, en:peanut-paste-min, en:p...",8606105235501,Štapići punjeni kikirikijem,"Marbo, Pardon, Pepsico_Štapići punjeni kikirik..."
3,"[{'lang': 'hr', 'text': 'Clipsy max'}, {'lang'...","Marbo,Pepsico",4.0,30.0,e,en,"[en:pellets, en:palm-oil, en:aroma-chicken, en...",8606105235549,Clipsy max,"Marbo,Pepsico_Clipsy max"
4,"[{'lang': 'main', 'text': 'Superseed snack clu...",Manitoba,3.0,17.0,d,en,"[en:hemp-seed, en:tapioca-syrup, en:sunflower-...",0697658201943,Superseed snack clusters,Manitoba_Superseed snack clusters


In [3]:
df.shape

(624303, 10)

In [13]:
df = df[~df['nutriscore_grade'].isin(['unknown', 'not-applicable'])]

### Feature Engineering
#### Ingredient parsing

In [17]:
# check type of input for ingredients_original_tags
print(df['ingredients_original_tags'].iloc[0])
print(type(df['ingredients_original_tags'].iloc[0]))

# Check the range of types present in the column
print(df['ingredients_original_tags'].apply(type).value_counts())

['en:peach-puree' 'en:coconut-milk' 'en:apple' 'en:strawberry'
 'en:pineapple' 'en:mango' 'en:banana' 'en:orange-juice'
 'en:butternut-squash' 'en:date' 'en:contains' 'en:water' 'en:coconut'
 'en:coconut']
<class 'numpy.ndarray'>
ingredients_original_tags
<class 'numpy.ndarray'>    441921
Name: count, dtype: int64


In [ ]:
# E number categories based on EU classification:
colorants = range(100,200) # E100-E199
preservatives = range(200,300) # E200-E299
antioxidants = range(300,400) # E300-E399
emulsifiers = range(400,500) # E400-E499
anti_caking_agents = range(500,600) # E500-E599
flavor_enhancers = range(600,700) # E600-E699
antibiotics = range(700,800) # E700-E799
sweeteners = range(900,1000) # E950-969
additional_categories = range(1000,1100) # E1000-E1100

In [19]:
# Extract E numbers integers from ingredient tags
def extract_e_number(tag):
    "Extract integer from tag and return none if not an E number"
    match = re.match(r'en:e(\d+)', tag)
    return int(match.group(1)) if match else None

In [20]:
# engineer additive features 
def engineer_additive_features(tags):
    " take the list of ingredient tag strings and return dict of additive features"
    e_nums = [n for tag in tags if (n := extract_e_number(tag)) is not None]

    return {
        "additive_count": len(e_nums),
        "has_additive": int(len(e_nums) > 0),
        "emulsifier_count": sum(1 for e in e_nums if e in emulsifiers),
        "antioxidants_count": sum(1 for e in e_nums if e in antioxidants),
        "preservatives_count": sum(1 for e in e_nums if e in preservatives),
        "colorants_count": sum(1 for e in e_nums if e in colorants),
        "sweeteners_count": sum(1 for e in e_nums if e in sweeteners),
    }

# apply and expand into columns
additive_features = df['ingredients_original_tags'].apply(engineer_additive_features)
df = pd.concat(
    [df, pd.DataFrame(additive_features.tolist(), index=df.index)],
    axis=1
)

In [21]:
# Remove E numbers from ingredients_original_tags to create a clean ingredient list
def remove_e_numbers(tags):
    """Join tags into a string, stripping E number tags to avoid double-encoding."""
    return ' '.join(
        t.replace('en:', '').replace('-', '_')   # clean for vectoriser vocab
        for t in tags
        if not re.match(r'en:e\d+', t)           # strip E numbers
    )

df['ingredients_clean'] = df['ingredients_original_tags'].apply(remove_e_numbers)

df['ingredients_clean'].head()

0    peach_puree coconut_milk apple strawberry pine...
1    red_bell_pepper sunflower_oil salt chili_peppe...
2    wheat_flour_min peanut_paste_min peanut dextro...
3    pellets palm_oil aroma_chicken potato_skrob po...
4    hemp_seed tapioca_syrup sunflower_seed brown_s...
Name: ingredients_clean, dtype: object

### Data Split
create df_train, df_val, df_test

In [22]:
# without nova 
missing_nova = df[df["nova_group"].isna()]
# with nova
with_nova = df[df["nova_group"].notna()]

# split with nova into stratified train, test, validation sets
from sklearn.model_selection import train_test_split

# Step 1: split off test (20%)
df_train_val, df_test = train_test_split(
    with_nova, 
    test_size=0.2, 
    stratify=with_nova['nova_group'], 
    random_state=42
)

# Step 2: split remaining 80% into train (70%) and val (10%)
# 0.125 of 80% = 10% of total
df_train, df_val = train_test_split(
    df_train_val, 
    test_size=0.125, 
    stratify=df_train_val['nova_group'], 
    random_state=42
)

print(f"Train: {len(df_train)} ({len(df_train)/len(with_nova):.0%})")
print(f"Val:   {len(df_val)} ({len(df_val)/len(with_nova):.0%})")
print(f"Test:  {len(df_test)} ({len(df_test)/len(with_nova):.0%})")

Train: 272139 (70%)
Val:   38877 (10%)
Test:  77754 (20%)


In [23]:
# Check stratification by nova group
print(df_train["nova_group"].value_counts())
print(df_test["nova_group"].value_counts())
print(df_val["nova_group"].value_counts())

nova_group
4.0    182141
3.0     54772
1.0     30455
2.0      4771
Name: count, dtype: int64
nova_group
4.0    52040
3.0    15649
1.0     8702
2.0     1363
Name: count, dtype: int64
nova_group
4.0    26020
3.0     7825
1.0     4351
2.0      681
Name: count, dtype: int64


The imbalance across the groups is preserved across all the splits which is good. Means they all reflect the same.

### Feature Extraction

In [24]:
# INGREDIENTS TAGS

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=500,
    min_df=20,        # ignore tags in fewer than 20 products
    binary=False,     # TF-IDF weights (down-weights en:salt etc.)
)

# Fit on training data
X_sparse_train = vectorizer.fit_transform(df_train['ingredients_clean'])

# Transform validation and test data using the same vectorizer
X_sparse_val   = vectorizer.transform(df_val['ingredients_clean'])
X_sparse_test  = vectorizer.transform(df_test['ingredients_clean'])

In [25]:
# NUTRIGRADE ENCODING TO NUMBERS
# encode nutriscore_grade as ordinal (a=0, b=1, c=2, d=3, e=4
from sklearn.preprocessing import OrdinalEncoder
import scipy.sparse as sp

grade_encoder = OrdinalEncoder(
    categories = [['a', 'b', 'c', 'd', 'e']]
)

def encode_grade(df_split):
    return grade_encoder.transform(df_split[['nutriscore_grade']])

# fit on training data
grade_encoder.fit(df_train[['nutriscore_grade']])

,categories,"[['a', 'b', ...]]"
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,unknown_value,None
,encoded_missing_value,nan
,min_frequency,None
,max_categories,None


In [26]:
# ADDITIVE FEATURES
additive_cols = [
    "additive_count",
    "has_additive",
    "emulsifier_count",
    "antioxidants_count",
    "preservatives_count",
    "colorants_count",
    "sweeteners_count"
]

In [27]:
# BUILD MATRIX
# obs: sparse_matrix in build_X is the TF-IDF output.
def build_feature_matrix(df_split, sparse_matrix):
    # encode grade
    nutriscore_grade = encode_grade(df_split)
    
    # extract additive features
    additive_features = df_split[additive_cols].fillna(0).values
    
    # combine sparse matrix with additive features
    return sp.hstack([nutriscore_grade, additive_features, sparse_matrix])

# apply function
X_train = build_feature_matrix(df_train, X_sparse_train) # X_spare_train is the TF-IDF output for the training set
X_val   = build_feature_matrix(df_val,   X_sparse_val)
X_test  = build_feature_matrix(df_test,  X_sparse_test)


print(f"X_train shape: {X_train.shape}")  # expect (n, 508) — 1 + 7 + 500

X_train shape: (272139, 515)


In [28]:
print(f"Grade:     1")
print(f"Additives: {len(additive_cols)}")
print(f"TF-IDF:    {X_sparse_train.shape[1]}")
print(f"Total:     {1 + len(additive_cols) + X_sparse_train.shape[1]}")

Grade:     1
Additives: 7
TF-IDF:    500
Total:     508


### Ordinal Logistic Regression 
Next up, we're doing the ordinal logistic regression.

Notes for the upcoming code:

- We evaluate on X_val, not X_train as we don't want to report training set performance as our result

- QWK is the primary metric here, not accuracy as it penalises predictions that are far off more than ones that are just one step wrong, which matters a lot for ordinal problems like NOVA (i.e. predicting 1 when true is 4 is much worse than predicting 3).

- A QWK above 0.7 would suggest the signal is strong enough to justify moving to a neural network
Thus, the ordinal logistic regression is our baseline — it's fast, interpretable, and tells us whether the features (nutriscore grade, additives, ingredients) actually carry signal for predicting NOVA group.

    - If QWK is already high (0.7+), it means the problem is "easy" for a linear model and a neural network might not add much. If QWK is low, it suggests the relationship is more complex and a neural network could learn patterns the logistic regression can't.

## Wondering what alpha means?

Alpha controls how much the model is penalised for being complex. Think of it as a "simplicity dial":

- High alpha → the model is forced to keep coefficients small and simple. Less risk of overfitting, but might underfit if the signal is subtle.

- Low alpha → the model is free to fit the training data more closely. More expressive, but can overfit.

- alpha=1.0 is just the default starting point — it's a reasonable middle ground. After you see your first results we can tune it if needed, but don't worry about it for now.

## QWK

- QWK stands for Quadratic Weighted Kappa. It measures how well the model's predictions agree with the true labels, but crucially it penalises big mistakes more than small ones.
For example:

- Predicting NOVA 2 when the true answer is NOVA 1 → small penalty (one step off)

- Predicting NOVA 4 when the true answer is NOVA 1 → large penalty (three steps off)

- Plain accuracy doesn't care about this distinction — a wrong answer is a wrong answer. QWK is therefore a much fairer metric for ordinal problems like yours. It ranges from 0 (no better than random) to 1 (perfect).



In [34]:
import mord
from sklearn.metrics import accuracy_score, cohen_kappa_score


# Target labels (nova_group: 1,2,3,4)
y_train = df_train['nova_group'].astype(int).values - 1  # logisticAT expects 0-indexed classes
y_val   = df_val['nova_group'].astype(int).values - 1

# Initialize and fit ordinal logistic regression
model = mord.LogisticAT(alpha=1.0)  # alpha is regularisation strength
model.fit(X_train, y_train)

# Predict on validation set
y_pred = model.predict(X_val)

# Evaluate
acc = accuracy_score(y_val, y_pred)
qwk = cohen_kappa_score(y_val, y_pred, weights='quadratic')

print(f"Accuracy:              {acc:.4f}")
print(f"Quadratic Weighted Kappa (QWK): {qwk:.4f}")


Accuracy:              0.9302
Quadratic Weighted Kappa (QWK): 0.9321


#### Cross validate alpha over a range

In [35]:
from sklearn.model_selection import cross_val_score
import numpy as np

alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

for alpha in alphas:
    m = mord.LogisticAT(alpha=alpha)
    scores = cross_val_score(m, X_train, y_train, cv=5, scoring='accuracy')
    print(f"alpha={alpha:<8} mean acc={scores.mean():.4f}  std={scores.std():.4f}")

alpha=0.001    mean acc=0.9360  std=0.0003
alpha=0.01     mean acc=0.9360  std=0.0004
alpha=0.1      mean acc=0.9349  std=0.0006
alpha=1.0      mean acc=0.9301  std=0.0007
alpha=10.0     mean acc=0.9089  std=0.0011
alpha=100.0    mean acc=0.8566  std=0.0011
